# Generate FF++ Augmentation Test Embeddings

Generate CLIP embeddings cho FF++ test split với test-time image augmentations.

Output `features` là mean-normalized embedding trung bình của nhiều augmented views cho từng sample. Mặc định notebook không lưu toàn bộ per-view tensor để tiết kiệm disk.

## Kaggle Setup

Clone repo và install package local.

In [ ]:
!git clone -b dev https://github.com/hoavien0110/training-free-tta-for-deepfake-detection.git /kaggle/working/training-free-tta-for-deepfake-detection
%cd /kaggle/working/training-free-tta-for-deepfake-detection
!pip install -q open-clip-torch
# !pip install -q -e . --no-deps

## Check Inputs

Nếu Kaggle slug khác, sửa path trong cell Run.

In [ ]:
!ls -lah /kaggle/input
!find /kaggle/input/datasets/jamestashvik/deepfakebench -maxdepth 2 -type f -name "*.csv" -print
!find /kaggle/input/ffpp-corruption-level-2 -maxdepth 3 -type d | sort | sed -n "1,100p"

## Run

Mặc định gen clean FF++ test + 4 corruption level 2, mỗi sample có `4` augmented views rồi lấy mean embedding.

Policy `weak`: random resized crop, horizontal flip, color jitter nhẹ. Nếu muốn lưu cả tensor per-view `[N, V, D]`, bỏ flag `--no-store-views`.

In [ ]:
!python training/generate_ffpp_augmentation_test_embeddings.py \
  --split test \
  --include-clean \
  --level 2 \
  --csv-path /kaggle/input/datasets/jamestashvik/deepfakebench/deepfakebench_dataset.csv \
  --deepfakebench-root /kaggle/input/datasets/jamestashvik/deepfakebench/DeepFakeBench \
  --corruption-root /kaggle/input/ffpp-corruption-level-2 \
  --corruptions color_contrast color_saturation gaussian_blur resize \
  --augmentation-policy weak \
  --views 4 \
  --no-store-views \
  --output-dir /kaggle/working/ffpp_augmented_test_features \
  --batch-size 16 \
  --num-workers 4 \
  --device auto \
  --use-amp \
  --trust-paths \
  --skip-existing

## Preview Outputs

In [ ]:
!find /kaggle/working/ffpp_augmented_test_features -maxdepth 1 -type f -name "*.pt" -print | sort

In [ ]:
import torch
from pathlib import Path

path = sorted(Path("/kaggle/working/ffpp_augmented_test_features").glob("*.pt"))[0]
payload = torch.load(path, map_location="cpu")
print(path)
print("features", tuple(payload["features"].shape))
print("labels", tuple(payload["labels"].shape))
print("features_views", None if "features_views" not in payload else tuple(payload["features_views"].shape))
print("augmentation_policy", payload.get("augmentation_policy"))
print("augmentation_views", payload.get("augmentation_views"))
print("counts", torch.bincount(payload["labels"].long(), minlength=2).tolist())

## Level 5 Variant

Nếu cần level 5 giống notebook `06`, dùng command dưới đây và sửa mount paths nếu slug khác.

In [ ]:
# !python training/generate_ffpp_augmentation_test_embeddings.py \
#   --split test \
#   --level 5 \
#   --csv-path /kaggle/input/datasets/jamestashvik/deepfakebench/deepfakebench_dataset.csv \
#   --deepfakebench-root /kaggle/input/datasets/jamestashvik/deepfakebench/DeepFakeBench \
#   --corruptions color_contrast color_saturation gaussian_blur resize \
#   --corruption-root-map \
#     color_contrast=/kaggle/input/ff-color-contrast-5 \
#     color_saturation=/kaggle/input/ff-color-saturation-5 \
#     gaussian_blur=/kaggle/input/ff-gaussian-blur-5 \
#     resize=/kaggle/input/ff-resize-5 \
#   --augmentation-policy weak \
#   --views 4 \
#   --no-store-views \
#   --output-dir /kaggle/working/ffpp_level5_augmented_test_features \
#   --batch-size 16 \
#   --num-workers 4 \
#   --device auto \
#   --use-amp \
#   --trust-paths \
#   --skip-existing